In [ ]:
import ee
import geemap
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import folium
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
ee.Authenticate()

In [ ]:
# Step 2a: Authenticate your Google account
ee.Authenticate()  # Follow the link, sign in, and paste the code

# Step 2b: Initialize EE with your project
ee.Initialize(project='neat-resolver-485318-e3')

In [ ]:
# Path to your shapefile
shapefile_path = "/content/drive/MyDrive/Hyderabad_LULC/training_points.shp"

# Load using GeoPandas
points_gdf = gpd.read_file(shapefile_path)

# Optional: check the first few points
points_gdf.head()

,id,Class,Label,geometry
0,1.0,5,water,POINT (78.47482 17.42486)
1,NaN,5,water,POINT (78.46911 17.42009)
2,NaN,5,water,POINT (78.47227 17.41498)
3,NaN,5,water,POINT (78.48233 17.42247)
4,NaN,5,water,POINT (78.46576 17.42797)


In [ ]:
# Convert all points in shapefile to EE FeatureCollection
points_fc = geemap.geopandas_to_ee(points_gdf)

# Define ROI as the convex hull (enclosing all points)
geometry_coords = [
    [78.23915715061369,17.540972162961793],
    [78.29408879123869,17.55144734245961],
    [78.30988163791838,17.558648677031712],
    [78.31949467502776,17.554066042706186],
    [78.33254093967619,17.56126727314872],
    [78.37648625217619,17.56126727314872],
    [78.410131882059,17.59465105201186],
    [78.44446415744963,17.609049994416633],
    [78.55982060276213,17.58417837304481],
    [78.60376591526213,17.575014281269635],
    [78.62367863498869,17.564540465033406],
    [78.66144413791838,17.49251658804457],
    [78.66899723850432,17.439462569701394],
    [78.68273014866057,17.4060502655658],
    [78.68273014866057,17.368699872713645],
    [78.65869755588713,17.333963725592014],
    [78.64908451877776,17.278896747447035],
    [78.56737370334807,17.223813301498527],
    [78.51724858127776,17.20807215582183],
    [78.41081852756682,17.202168880800254],
    [78.38609928928557,17.20807215582183],
    [78.38060612522307,17.23889730818473],
    [78.37373967014494,17.26316029841092],
    [78.38197941623869,17.28283064969935],
    [78.36824650608244,17.336585551102786],
    [78.34558720432463,17.388358924638855],
    [78.32910771213713,17.40736066716505],
    [78.27966923557463,17.415878048500176],
    [78.25494999729338,17.438807485270484],
    [78.24739689670744,17.47614354047107],
    [78.23297734104338,17.529186862422012],
    [78.23915715061369,17.540972162961793]
]

# Create EE Polygon and assign to `roi`
roi = ee.Geometry.Polygon([geometry_coords])

# Visualize the ROI
m = geemap.Map(center=[17.445, 78.525], zoom=12)
m.addLayer(roi, {'color': 'red'}, "Custom ROI")
m

Map(center=[17.445, 78.525], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchData…

In [ ]:
def vectorMap(layer, viz_params, name):
    """
    Display a vector layer on a geemap map.

    Args:
        layer (ee.FeatureCollection or ee.Geometry): The layer to display
        viz_params (dict): Visualization parameters, e.g., color
        name (str): Layer name
    """
    m = geemap.Map(center=[17.445, 78.525], zoom=12)  # Hyderabad ORR
    m.add_basemap("SATELLITE")
    m.addLayer(layer, viz_params, name, opacity=0.5)
    return m

In [ ]:
vectorMap(roi, {"fillColor": "00000000", "color": "FF0000"}, "ORR Boundary")

Map(center=[17.445, 78.525], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchData…

In [ ]:
# Set the start and end dates to search for images
start_date = "2023-02-01"
end_date = "2023-11-30"

In [ ]:
def s2Process(dates, extent, cloud):
    """
    Download and process Sentinel-2 images over the given extent and date range.
    """

    start_date, end_date = dates

    # Load and filter Sentinel-2 image collection
    def imgDownload():
        img_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                          .filterDate(start_date, end_date)
                          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud))
                          .filterBounds(extent))
        print(f"There are {img_collection.size().getInfo()} Sentinel-2 Images")
        return img_collection

    # Cloud masking
    def s2ClearSky(image):
        scl = image.select('SCL')
        clear_sky_pixels = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(11))
        return image.updateMask(clear_sky_pixels).multiply(0.0001)

    # Resample 20m bands to 10m
    def bandResample(image):
        bands_10m = image.select(["B2", "B3", "B4", "B8"])
        bands_20m = image.select(["B5","B6","B7","B8A","B11","B12"]) \
                         .resample("bilinear") \
                         .reproject(crs=bands_10m.projection(), scale=10)
        return bands_10m.addBands(bands_20m)

    # Compute spectral indices
    def indices(image):
        ndvi = image.normalizedDifference(["B8", "B4"]).rename("ndvi")
        mndwi = image.normalizedDifference(["B3", "B11"]).rename("mndwi")
        ndbi = image.normalizedDifference(["B11", "B8"]).rename("ndbi")
        bsi = image.expression(
            "((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))",
            {
                "SWIR1": image.select("B11"),
                "RED": image.select("B4"),
                "NIR": image.select("B8"),
                "BLUE": image.select("B2"),
            }
        ).rename("bsi")
        return image.addBands([ndvi, mndwi, ndbi, bsi])

    # Download image collection
    s2_collection = imgDownload()

    # Apply cloud mask, resample bands, compute indices
    s2_collection_resampled = (s2_collection
                               .map(s2ClearSky)
                               .map(bandResample)
                               .map(indices))

    # Compute median composite and clip to extent
    s2_median = s2_collection_resampled.median().clip(extent)

    # Get projection info using a specific band
    s2_median_projection = s2_median.select("B4").projection()
    s2_median_crs = s2_median_projection.crs().getInfo()
    s2_median_scale = s2_median_projection.nominalScale().getInfo()
    print(f"The CRS of s2_median is {s2_median_crs}")
    print(f"The spatial resolution of s2_median is {s2_median_scale}")

    # Check band names
    s2_band_names = s2_median.bandNames().getInfo()
    print(f"The bands in 's2_median': {s2_band_names}")

    return s2_median

In [ ]:
dates = (start_date, end_date)      # ('2023-02-01', '2023-11-30')
roi                                 # ORR ROI from shapefile / convex hull
cloud = 30
s2_median_2023 = s2Process(dates, roi, cloud)                     # maximum cloud percentage

There are 61 Sentinel-2 Images
The CRS of s2_median is EPSG:4326
The spatial resolution of s2_median is 111319.49079327357
The bands in 's2_median': ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'ndvi', 'mndwi', 'ndbi', 'bsi']


In [ ]:
def rasterMap(layer, viz_params, name):
    """
    Display a raster layer (ee.Image) on a geemap map.
    """
    m = geemap.Map(center=[17.445, 78.525], zoom=12)  # Hyderabad ORR
    m.add_basemap("SATELLITE")
    m.addLayer(layer, viz_params, name)   # <-- just pass viz_params, no viz_params=
    return m

In [ ]:
s2_vis_params = {"min":0, "max": 0.3, "bands":["B8", "B4", "B3"], "gamma":0.6}

rasterMap(s2_median_2023, s2_vis_params, "S2 Median Composite 2023")

Map(center=[17.445, 78.525], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchData…

In [ ]:
def s1Process(dates, extent, radius):
    """
    Download and process Sentinel-1 GRD images over the given extent and date range.
    """

    start_date, end_date = dates

    # Load and filter Sentinel-1 collection
    def imgdownload():
        img_collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                          .filterDate(start_date, end_date)
                          .filter(ee.Filter.eq("instrumentMode", "IW"))
                          .filterMetadata("resolution_meters", "equals", 10)
                          .filterBounds(extent)
                          .select(["VV", "VH"]))

        print(f"There are {img_collection.size().getInfo()} Sentinel-1 Images")
        polarizations = img_collection.first().bandNames().getInfo()
        print(f"Polarizations in the collection: {polarizations}")
        return img_collection

    # Apply speckle filter
    def speckle(image):
        image_filtered = image.focal_median(radius, "circle", "meters")
        return image_filtered.copyProperties(image, image.propertyNames())

    # Compute VH / VV ratio
    def ratio(image):
        vh = image.select("VH")
        vv = image.select("VV")
        ratio_band = vh.divide(vv).rename("VH_VV_ratio")
        return image.addBands([ratio_band])

    # Download collection
    s1_collection = imgdownload()

    # Apply speckle filter and compute ratio
    s1_collection_filtered = s1_collection.map(speckle).map(ratio)

    # Median composite
    s1_median = s1_collection_filtered.median().clip(extent)

    # Projection info using VV band
    s1_median_projection = s1_median.select("VV").projection()
    s1_median_crs = s1_median_projection.crs().getInfo()
    s1_median_scale = s1_median_projection.nominalScale().getInfo()

    print(f"CRS of s1_median: {s1_median_crs}")
    print(f"Spatial resolution of s1_median: {s1_median_scale}")

    # Bands
    s1_band_names = s1_median.bandNames().getInfo()
    print(f"Bands in 's1_median': {s1_band_names}")

    return s1_median

In [ ]:
# Download s1-image collectionn, apply speckle filter, compute ratio, and median composites
dates = (start_date, end_date)
s1_median_2023 = s1Process(dates, roi, 50)

# Select different polarizations from s1_median_2023
s1_median_2023_VH = s1_median_2023.select("VH")
s1_median_2023_VV = s1_median_2023.select("VV")
s1_median_2023_ratio = s1_median_2023.select("VH_VV_ratio")

There are 25 Sentinel-1 Images
Polarizations in the collection: ['VV', 'VH']
CRS of s1_median: EPSG:4326
Spatial resolution of s1_median: 111319.49079327357
Bands in 's1_median': ['VV', 'VH', 'VH_VV_ratio']


In [ ]:
def linkedMap(layers, viz_params, labels):
    """
    Display 4 layers in a 2x2 linked map for comparison
    layers: list of ee.Image
    viz_params: list of dicts, visualization parameters for each layer
    labels: list of strings, labels for each layer
    """
    # Use ORR ROI center
    center_coords = [17.445, 78.525]  # ORR approximate center

    map = geemap.linked_maps(
        rows=2,
        cols=2,
        height="450px",
        center=center_coords,
        zoom=12,
        ee_objects=layers,
        vis_params=viz_params,
        labels=labels,
        label_position="topright",
    )
    return map

In [ ]:
dates = (start_date, end_date)
s1_median_2023 = s1Process(dates, roi, 50)

s1_median_2023_VH = s1_median_2023.select("VH")
s1_median_2023_VV = s1_median_2023.select("VV")
s1_median_2023_ratio = s1_median_2023.select("VH_VV_ratio")

There are 25 Sentinel-1 Images
Polarizations in the collection: ['VV', 'VH']
CRS of s1_median: EPSG:4326
Spatial resolution of s1_median: 111319.49079327357
Bands in 's1_median': ['VV', 'VH', 'VH_VV_ratio']


In [ ]:
def linkedMap(layers, viz_params, labels, roi):
    """
    Display a linked map of multiple layers centered on the ROI.
    """
    # Compute center from ROI
    roi_center = roi.centroid().coordinates().getInfo()[::-1]  # [lat, lon]

    map = geemap.linked_maps(
        rows=2,
        cols=2,
        height="450px",
        center=roi_center,
        zoom=12,
        ee_objects=layers,
        vis_params=viz_params,
        labels=labels,
        label_position="topright",
    )
    return map

In [ ]:
stacked_layer = s2_median_2023.addBands(s1_median_2023_VH).addBands(s1_median_2023_VV).addBands(s1_median_2023_ratio)
print(f"The bands in the stacked layer are: {stacked_layer.bandNames().getInfo()}")

The bands in the stacked layer are: ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'ndvi', 'mndwi', 'ndbi', 'bsi', 'VH', 'VV', 'VH_VV_ratio']


In [ ]:
# Check the properties of the "stacked_layer"

# Get the projection information from one of the bands
stacked_layer_projection = stacked_layer.select(16).projection()

# Get the CRS and spatial resolution
stacked_layer_crs = stacked_layer_projection.crs().getInfo()
stacked_layer_scale = stacked_layer_projection.nominalScale().getInfo()

print(f"The CRS of the stacked_layer is {stacked_layer_crs}")
print(f"The spatial resolution of the stacked_layer is {stacked_layer_scale}")

The CRS of the stacked_layer is EPSG:4326
The spatial resolution of the stacked_layer is 111319.49079327357


In [ ]:
# Load the training and testing data
sample_gdf = gpd.read_file("/content/drive/MyDrive/Hyderabad_LULC/training_points.shp")

sample_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   id        1 non-null      float64 
 1   Class     334 non-null    int64   
 2   Label     334 non-null    object  
 3   geometry  334 non-null    geometry
dtypes: float64(1), geometry(1), int64(1), object(1)
memory usage: 10.6+ KB


In [ ]:
sample_gdf["Class"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 334 entries, 0 to 333
Series name: Class
Non-Null Count  Dtype
--------------  -----
334 non-null    int64
dtypes: int64(1)
memory usage: 2.7 KB


In [ ]:
# Check the training samples
sample_gdf.head()

,id,Class,Label,geometry
0,1.0,5,water,POINT (78.47482 17.42486)
1,NaN,5,water,POINT (78.46911 17.42009)
2,NaN,5,water,POINT (78.47227 17.41498)
3,NaN,5,water,POINT (78.48233 17.42247)
4,NaN,5,water,POINT (78.46576 17.42797)


In [ ]:
# Visualize the samples on a map and symbolize them with different colour for the unique ID

palette = {
    1: "#c4734a",
    2: "#e31a1c",
    3: "#fae500",
    4: "#219f45",
    5: "#1452d9"}

# Create a dictionary to map IDs to colors
unique_ids = np.sort(sample_gdf["Class"].unique())
id_color_map = {unique_id: palette.get(unique_id, "#808080") for unique_id in unique_ids}

# Initialize the folium map
center = sample_gdf.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13)

# Add satellite basemap
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google Satellite",
    name="Google Satellite",
    overlay=True,
).add_to(m)

# Add points to the map
for idx, row in sample_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=id_color_map[row["Class"]],
        fill=True,
        fill_color=id_color_map[row["Class"]],
        fill_opacity=0.6,
        popup=f"ID: {row['Class']}, Class: {row['Label']}"
    ).add_to(m)

# Display the map
m

In [ ]:
# Sample Class Distribution
count = sample_gdf["Class"].value_counts().sort_values(ascending=False)
per = ((count / len(sample_gdf)) * 100).round(2)

# Create a DataFrame for better display
count_per = pd.DataFrame({
    "Class": count.index,
    "Count": count.values,
    "Percentage": per.values
})

print(count_per)

   Class  Count  Percentage
0      2     90       26.95
1      4     72       21.56
2      5     61       18.26
3      3     60       17.96
4      1     51       15.27


In [ ]:
sample_ee_all = geemap.gdf_to_ee(sample_gdf)
print(sample_ee_all.first().getInfo())

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [78.47482242137498, 17.42486447760898]}, 'id': '0', 'properties': {'Class': 5, 'Label': 'water', 'id': 1}}


In [ ]:
stacked_layer_small = stacked_layer.select([
    "B2", "B3", "B4", "B8",   # Sentinel-2 visible/NIR bands
    "VH", "VV", "VH_VV_ratio"  # Sentinel-1 bands
])

In [ ]:
import ee
import geemap

# 1️⃣ Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='neat-resolver-485318-e3')

# 2️⃣ Select only essential bands to reduce memory usage
stacked_layer_small = stacked_layer.select([
    "B2", "B3", "B4", "B8",  # Sentinel-2 visible/NIR
    "VH", "VV", "VH_VV_ratio"  # Sentinel-1
])

# 3️⃣ Define a memory-friendly train/test split function
def trainTestSmall(sample, image, threshold=0.7):
    """
    Split samples into training and testing sets and extract image values.
    """
    # Add random column for splitting
    sample_random = sample.randomColumn("random")

    # Training and testing sets
    training_set = sample_random.filter(ee.Filter.lt("random", threshold))
    testing_set = sample_random.filter(ee.Filter.gte("random", threshold))

    # Extract spectral properties to training points
    training = image.sampleRegions(
        collection=training_set,
        properties=["Class"],
        scale=20  # coarser scale to save memory
    )

    # Print info
    print(f"Training set size: {training.size().getInfo()}")
    print(f"Testing set size: {testing_set.size().getInfo()}")
    print("Training Example:", training.first().getInfo())
    print("Testing Example:", testing_set.first().getInfo())

    return training, training_set, testing_set

# 4️⃣ Convert your sample shapefile to EE
sample_ee_all = geemap.geopandas_to_ee(points_gdf)

# 5️⃣ Apply the small train/test function
training, training_set, testing_set = trainTestSmall(sample_ee_all, stacked_layer_small, 0.7)

Training set size: 226
Testing set size: 105
Training Example: {'type': 'Feature', 'geometry': None, 'id': '0_0', 'properties': {'B2': 0.0685, 'B3': 0.08945, 'B4': 0.06440000000000001, 'B8': 0.06115, 'Class': 5, 'VH': -26.890493540002307, 'VH_VV_ratio': 1.243523596379611, 'VV': -21.719063014966217}}
Testing Example: {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [78.44634687766246, 17.338901278856206]}, 'id': '9', 'properties': {'Class': 5, 'Label': 'water', 'random': 0.9425386517163108}}


In [ ]:
# Compute center from your ROI
roi_center = roi.centroid().coordinates().getInfo()[::-1]  # [lat, lon]

# Initialize map
Map = geemap.Map(center=roi_center, zoom=12)

# Style dictionaries
training_style = {"color": "blue", "pointSize": 3}
testing_style = {"color": "red", "pointSize": 3}

# Style training and testing samples
training_layer = training_set.style(**training_style)
testing_layer = testing_set.style(**testing_style)

# Add layers
Map.addLayer(training_layer, {}, "Training Samples")
Map.addLayer(testing_layer, {}, "Testing Samples")

# Display
Map

Map(center=[17.41526543184839, 78.4808179401501], controls=(WidgetControl(options=['position', 'transparent_bg…

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import ee

def rf(parameters, image, training, testing, out_folder, out_file_base):
    """
    Train, classify, and validate using Random Forest in GEE.

    parameters: tuple (trees, variablesPerSplit, minLeafPopulation, bagFraction, maxNodes, seed)
    image: ee.Image with bands stacked
    training: ee.FeatureCollection of training points
    testing: ee.FeatureCollection of testing points
    out_folder: folder to save CSVs
    out_file_base: base name for CSVs
    """

    if not os.path.exists(out_folder):
        os.mkdir(out_folder)

    # CSV paths
    error_file_name = os.path.join(out_folder, out_file_base + "error_matrix.csv")
    importance_file_name = os.path.join(out_folder, out_file_base + "feature_importance.csv")

    # Unpack RF parameters
    trees, split, leaf, bag, node, seed = parameters

    # Instantiate RF
    model = ee.Classifier.smileRandomForest(
        numberOfTrees=trees,
        variablesPerSplit=split,
        minLeafPopulation=leaf,
        bagFraction=bag,
        maxNodes=node,
        seed=seed
    )

    # List of bands present in training points
    bands_for_training = ['B2','B3','B4','B8','VH','VV','VH_VV_ratio']

    # Train classifier
    classifier = model.train(
        features=training,
        classProperty="Class",
        inputProperties=bands_for_training
    )

    # Classify the image
    classified = image.select(bands_for_training).classify(classifier)

    # Validation
    validation = classified.sampleRegions(
        collection=testing,
        properties=["Class"],
        scale=10
    )

    # Error matrix
    error_matrix = validation.errorMatrix(actual="Class", predicted="classification")

    # Prepare confusion matrix DataFrame
    error_matrix_array = error_matrix.array().getInfo()
    data = [row[1:] for row in error_matrix_array[1:]]
    columns = ["bare-soil", "built-up", "cropland", "tree&grass", "water"]
    error_matrix_df = pd.DataFrame(data, columns=columns)

    # Print accuracies
    print("Confusion matrix for Random Forest:\n", error_matrix_df)
    print("Overall Accuracy:", round(error_matrix.accuracy().getInfo(),3))
    print("Producer's Accuracy:", error_matrix.producersAccuracy().getInfo())
    print("Consumer's Accuracy:", error_matrix.consumersAccuracy().getInfo())
    print("Kappa:", round(error_matrix.kappa().getInfo(),3))

    # Feature importance
    explained = classifier.explain()
    importance = ee.Dictionary(explained.get("importance"))
    feature_names = importance.keys().getInfo()
    importance_values = importance.values().getInfo()

    importance_combined = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance_values
    })

    # Normalize
    scaler = MinMaxScaler()
    importance_values_norm = scaler.fit_transform([[v] for v in importance_combined['Importance']])
    importance_values_norm_percent = (importance_values_norm / importance_values_norm.sum()) * 100
    importance_values_percent = (importance_combined['Importance'] / importance_combined['Importance'].sum()) * 100

    importance_combined['Importance_Norm'] = importance_values_norm.flatten()
    importance_combined['Importance_Norm_Percent'] = importance_values_norm_percent.flatten()
    importance_combined['Importance_Percent'] = importance_values_percent

    # Save CSV
    importance_combined.to_csv(importance_file_name, index=False)

    return classified, error_file_name, importance_file_name

In [ ]:
# RF default parameters
rf_def_parameters = (10, None, 1, 0.5, None, 0)

# Apply RF
rf_classified, rf_confusion_csv, rf_importance_csv = rf(
    rf_def_parameters,
    stacked_layer,
    training,    # 229 training samples
    testing_set,     # 105 testing samples
    "Outputs",
    "Hyderabad_RF_"
)

Confusion matrix for Random Forest:
    bare-soil  built-up  cropland  tree&grass  water
0          8         0         0           4      0
1          0        25         0           1      0
2          4         0        15           1      0
3          0         0         2          22      0
4          1         0         0           0     20
Overall Accuracy: 0.874
Producer's Accuracy: [[0], [0.6666666666666666], [0.9615384615384616], [0.75], [0.9166666666666666], [0.9523809523809523]]
Consumer's Accuracy: [[0, 0.6153846153846154, 1, 0.8823529411764706, 0.7857142857142857, 1]]
Kappa: 0.84


In [ ]:
def sliderMap(layer1params, layer2params):
    """
    Create a split slider map for visual comparison
    """

    ee_object1, vis_params1, name1 = layer1params
    ee_object2, vis_params2, name2 = layer2params

    left_layer = geemap.ee_tile_layer(ee_object1, vis_params1, name=name1)
    right_layer = geemap.ee_tile_layer(ee_object2, vis_params2, name=name2)

    # ✅ Correct Hyderabad center (lat, lon)
    m = geemap.Map(center=[17.3850, 78.4867], zoom=12)

    m.split_map(left_layer, right_layer)
    m.addLayerControl()

    return m

In [ ]:
# Visualize RF Land Cover vs Sentinel-2 using slider map

# Land cover visualization parameters (5 classes)
lc_vis_params = {
    "min": 1,
    "max": 5,
    "palette": [
        "#c4734a",  # bare-soil
        "#e31a1c",  # built-up
        "#fae500",  # cropland
        "#219f45",  # tree & grass
        "#1452d9"   # water
    ]
}

# Sentinel-2 RGB visualization
s2_vis_params_rgb = {
    "min": 0,
    "max": 0.3,
    "bands": ["B4", "B3", "B2"],
    "gamma": 0.7
}

# Left panel: Sentinel-2
s2_params = (
    s2_median_2023,
    s2_vis_params_rgb,
    "Sentinel-2 (Median)"
)

# Right panel: RF classification  ✅ USE YOUR VARIABLE
rf_lc_params = (
    rf_classified,          # <-- NOT rf_def_classification
    lc_vis_params,
    "Land Cover (RF DEFAULT)"
)

# Slider comparison map
sliderMap(s2_params, rf_lc_params)

Map(center=[17.385, 78.4867], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [ ]:
# Create the slider map and capture the returned folium map
slider_map = sliderMap(s2_params, rf_lc_params)

# Function to add a legend
def add_legend(map_obj, title, labels, colors):
    legend_html = f'''
    <div style="
        position: fixed;
        bottom: 50px; left: 50px; width: 150px; height: 180px;
        background-color: white;
        border:2px solid grey;
        z-index:9999;
        font-size:14px;
        padding: 10px;
        "><b>{title}</b><br>
    '''
    for label, color in zip(labels, colors):
        legend_html += f'''
        <i style="background:{color}; width:12px; height:12px; float:left; margin-right:5px;"></i>
        {label}<br>
        '''
    legend_html += '</div>'
    map_obj.get_root().html.add_child(folium.Element(legend_html))

# Add legend to the slider map
add_legend(slider_map, "Land Cover Classes", class_names, class_colors)

# Display the map
slider_map


RF: Hyper-Parameter Tuning

In [ ]:
# Apply the rf function using tuned parameters (balanced & stable)

# Tuned for Sentinel-2 + limited training samples
# (Better than aggressive 1000 trees, depth 7)
rf_tun_parameters = (300, 5, None, None, None, None)

# Run tuned Random Forest
rf_tun_classified, rf_tun_confusion_path, rf_tun_fea_importance_path = rf(
    rf_tun_parameters,
    stacked_layer,
    training,        # 229 training samples
    testing_set,     # 105 testing samples
    "Outputs",
    "rf_tun_classification_"
)

Confusion matrix for Random Forest:
    bare-soil  built-up  cropland  tree&grass  water
0          8         0         1           3      0
1          0        25         0           1      0
2          4         0        14           2      0
3          0         0         2          22      0
4          0         0         0           0     21
Overall Accuracy: 0.874
Producer's Accuracy: [[0], [0.6666666666666666], [0.9615384615384616], [0.7], [0.9166666666666666], [1]]
Consumer's Accuracy: [[0, 0.6666666666666666, 1, 0.8235294117647058, 0.7857142857142857, 1]]
Kappa: 0.84


In [ ]:
# First, make sure points_fc is defined
# If you have points_gdf from earlier:
points_fc = geemap.geopandas_to_ee(points_gdf)

# Then create ROI
roi = points_fc.geometry().convexHull()

# Now get bounds
roi_bounds = roi.bounds().getInfo()['coordinates'][0]
print(f"ROI bounds coordinates: {roi_bounds}")

# Convert to [west, south, east, north]
west = min(coord[0] for coord in roi_bounds)
south = min(coord[1] for coord in roi_bounds)
east = max(coord[0] for coord in roi_bounds)
north = max(coord[1] for coord in roi_bounds)

roi_export = [west, south, east, north]
print(f"Export bounds: {roi_export}")

ROI bounds coordinates: [[78.2372131819217, 17.24540691655833], [78.67755556590002, 17.24540691655833], [78.67755556590002, 17.566047110823863], [78.2372131819217, 17.566047110823863], [78.2372131819217, 17.24540691655833]]
Export bounds: [78.2372131819217, 17.24540691655833, 78.67755556590002, 17.566047110823863]


In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np

# Initialize
ee.Initialize(project='neat-resolver-485318-e3')

# 1. LOAD YOUR POINTS (replace with your actual path)
# If you have a shapefile:
points_gdf = gpd.read_file("/content/drive/MyDrive/Hyderabad_LULC/training_points.shp")

# Convert to EE FeatureCollection
points_fc = geemap.geopandas_to_ee(points_gdf)

# 2. CREATE ROI
roi = points_fc.geometry().convexHull()

# 3. GET BOUNDS FOR EXPORT
def get_export_bounds(geometry):
    """Get bounds from any geometry for export"""

    # Get coordinates
    coords_info = geometry.coordinates().getInfo()

    # Handle different geometry types
    if isinstance(coords_info[0][0], (int, float)):
        # Single point or simple polygon
        flat_coords = coords_info[0] if isinstance(coords_info[0][0], list) else coords_info
    else:
        # Multi-polygon or complex geometry
        flat_coords = []
        for ring in coords_info:
            if isinstance(ring[0], list):
                flat_coords.extend(ring)
            else:
                flat_coords.append(ring)

    # Get min/max
    lons = [c[0] for c in flat_coords]
    lats = [c[1] for c in flat_coords]

    # Add 5% buffer
    lon_buffer = (max(lons) - min(lons)) * 0.05
    lat_buffer = (max(lats) - min(lats)) * 0.05

    return [
        min(lons) - lon_buffer,
        min(lats) - lat_buffer,
        max(lons) + lon_buffer,
        max(lats) + lat_buffer
    ]

# Get export bounds
export_bounds = get_export_bounds(roi)
print(f"Export bounds: {export_bounds}")

# 4. CREATE YOUR STACKED IMAGE (use your existing functions)
dates = ('2023-02-01', '2023-11-30')

# Get Sentinel-2 (from your s2Process function)
s2_median_2023 = s2Process(dates, roi, 30)

# Get Sentinel-1 (from your s1Process function)
s1_median_2023 = s1Process(dates, roi, 50)

# Select VH and VV bands
s1_VH = s1_median_2023.select("VH")
s1_VV = s1_median_2023.select("VV")
s1_ratio = s1_median_2023.select("VH_VV_ratio")

# Create 7-band stack
stacked_layer_small = s2_median_2023.select(['B2', 'B3', 'B4', 'B8'])\
    .addBands(s1_VH)\
    .addBands(s1_VV)\
    .addBands(s1_ratio)

print(f"Bands in stacked image: {stacked_layer_small.bandNames().getInfo()}")

# 5. EXPORT
export_task = ee.batch.Export.image.toDrive(
    image=stacked_layer_small,
    description='Hyderabad_7bands_Patches',
    folder='Hyderabad_Patches',
    fileNamePrefix='hyderabad_7bands',
    scale=10,
    region=export_bounds,
    maxPixels=2e9,
    crs='EPSG:4326',
    fileFormat='GeoTIFF'
)

export_task.start()

print(f"""
✅ EXPORT STARTED!
Task ID: {export_task.id}
File: hyderabad_7bands.tif
Location: Google Drive → 'Hyderabad_Patches' folder

Bounds used: {export_bounds}

Check status: https://code.earthengine.google.com/tasks
""")

# Show a quick map to verify
m = geemap.Map(center=[17.445, 78.525], zoom=12)
m.addLayer(roi, {'color': 'red'}, 'ROI')
m.addLayer(stacked_layer_small, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'Image')
m

Export bounds: [78.21519606272278, 17.22937490684508, 78.69957268509894, 17.58207912053711]
There are 61 Sentinel-2 Images
The CRS of s2_median is EPSG:4326
The spatial resolution of s2_median is 111319.49079327357
The bands in 's2_median': ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'ndvi', 'mndwi', 'ndbi', 'bsi']
There are 25 Sentinel-1 Images
Polarizations in the collection: ['VV', 'VH']
CRS of s1_median: EPSG:4326
Spatial resolution of s1_median: 111319.49079327357
Bands in 's1_median': ['VV', 'VH', 'VH_VV_ratio']
Bands in stacked image: ['B2', 'B3', 'B4', 'B8', 'VH', 'VV', 'VH_VV_ratio']

✅ EXPORT STARTED!
Task ID: LZ53NCCWV6NEL3GFULVYJKLL
File: hyderabad_7bands.tif
Location: Google Drive → 'Hyderabad_Patches' folder

Bounds used: [78.21519606272278, 17.22937490684508, 78.69957268509894, 17.58207912053711]

Check status: https://code.earthengine.google.com/tasks



Map(center=[17.445, 78.525], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchData…